# KhaledDSML — عرض الوصف الكامل في Google Colab

يعرض هذا الدفتر أحدث وصف من GitHub، مع بطاقة المؤلف (الاسم واللقب والصفة الأكاديمية والجامعة والكلية)، واتجاه عربي **RTL** وإنجليزي **LTR**. تُعرض أمثلة البرمجة دائمًا من اليسار.

Notebook renders the **latest GitHub README**, not the immutable PyPI 0.10.0 snapshot. Internet access is required.


In [ ]:
%pip -q install markdown beautifulsoup4 requests

In [ ]:
import re
import requests
import markdown
from bs4 import BeautifulSoup
from IPython.display import HTML, display

# Always read the *latest* README on GitHub.
readme_url = "https://raw.githubusercontent.com/noble2013/KhaledDSML/main/README.md"
response = requests.get(readme_url, timeout=30)
response.raise_for_status()

# Convert Markdown to real HTML (tables, headings, fenced Python examples).
rendered = markdown.markdown(
    response.text,
    extensions=["fenced_code", "tables", "sane_lists", "toc"],
    output_format="html5",
)
soup = BeautifulSoup(rendered, "html.parser")

# Arabic paragraphs/headings -> RTL right; English -> LTR left.
# Keep existing explicit dir attributes in the author profile.
for tag in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p", "li", "th", "td", "blockquote"]):
    if tag.find_parent(["pre", "code"]) or tag.has_attr("dir"):
        continue
    value = tag.get_text(" ", strip=True)
    arabic_count = len(re.findall(r"[\u0621-\u064a]", value))
    latin_count = len(re.findall(r"[A-Za-z]", value))
    rtl = arabic_count > 0 and arabic_count >= latin_count * 0.65
    tag["dir"] = "rtl" if rtl else "ltr"
    tag["lang"] = "ar" if rtl else "en"
    tag["style"] = "text-align: right;" if rtl else "text-align: left;"

# Programming code must remain LTR even in an Arabic section.
for code_tag in soup.find_all(["pre", "code"]):
    code_tag["dir"] = "ltr"
    code_tag["style"] = "direction: ltr; text-align: left;"

style = """
<style>
.kdsml-doc {
  background: #fff; color: #18233c; max-width: 1040px;
  margin: 12px auto; padding: 24px 30px; border-radius: 18px;
  border: 1px solid #dfe5ee; line-height: 1.85;
  font-family: Tahoma, Arial, sans-serif; font-size: 16px;
}
.kdsml-doc h1,.kdsml-doc h2,.kdsml-doc h3 {
  color: #173f77; border-bottom: 1px solid #e2e9f4;
  padding-bottom: 8px; margin-top: 24px;
}
.kdsml-doc h1 {font-size: 28px}
.kdsml-doc h2 {font-size: 23px}
.kdsml-doc a {color: #1469b2}
.kdsml-doc table {border-collapse: collapse; margin: 15px 0; width: 100%}
.kdsml-doc th,.kdsml-doc td {border: 1px solid #d4deed; padding: 10px}
.kdsml-doc th {background: #f0f5fb}
.kdsml-doc pre {
  direction: ltr !important; text-align: left !important;
  white-space: pre; overflow-x: auto; background: #f1f5f9;
  color: #1e293b; padding: 14px; border-radius: 10px;
}
.kdsml-doc pre code {font-family: Consolas, monospace}
.kdsml-doc img {max-width: 100%; height: auto}
</style>
"""
display(HTML(style + '<div class="kdsml-doc" lang="ar" dir="rtl">' + str(soup) + '</div>'))
